# Process Mining Group 5 notebook

## Preprocessing & data quality

Applied in the discovery cell before mining:

1. Keep only `complete` lifecycle events.
2. Drop exact duplicate `(case, activity, timestamp)` rows.
3. Optional rare-activity filter (`MIN_ACTIVITY_FREQ`).
4. Scope mining per object flow (Permit, Declaration, Request For Payment) to avoid a spaghetti model.
5. Keep only complete cases (last activity contains `FINAL_APPROVED` or `REJECTED`).


## Discovery & evaluation

- Miners: Heuristics and Inductive
- Metrics: fitness, precision, generalization, simplicity, plus F-score and their mean.
- Each sub-process is mined on samples and on all of its cases; nets are saved as PNML and the best per sub-process as PNG.


In [4]:
import os
import re
import pandas as pd
import pm4py

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"
LIFECYCLE = "lifecycle:transition"

XES_PATH = "PermitLog.xes"
OUTPUT_FOLDER = "Output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
log = pm4py.read_xes(XES_PATH)
df = pm4py.convert_to_dataframe(log)

if LIFECYCLE in df.columns:
    complete_mask = df[LIFECYCLE].astype(str).str.lower().eq("complete")
    if complete_mask.any():
        df = df[complete_mask].copy()

df = df[[CASE_ID, ACTIVITY, TIMESTAMP]].dropna().copy()
df[TIMESTAMP] = pd.to_datetime(df[TIMESTAMP], errors="coerce", utc=True)
df = df.dropna(subset=[TIMESTAMP])
df = df.sort_values([CASE_ID, TIMESTAMP, ACTIVITY]).reset_index(drop=True)

act_freq = df[ACTIVITY].value_counts()
df = df.drop_duplicates(subset=[CASE_ID, ACTIVITY, TIMESTAMP]).copy()

df = df.sort_values([CASE_ID, TIMESTAMP, ACTIVITY]).reset_index(drop=True)

SUBPROCESSES = {
    "permit":              {"prefixes": ["Permit"],              "end": ["FINAL_APPROVED", "REJECTED"]},
    "declaration":         {"prefixes": ["Declaration"],         "end": ["FINAL_APPROVED", "REJECTED"]},
    "request_for_payment": {"prefixes": ["Request For Payment"], "end": ["FINAL_APPROVED", "REJECTED"]},
}

BASE_SAMPLE_SIZES = [250, 500, 1000]

heuristics_settings = [
    {"dependency_threshold": 0.20, "and_threshold": 0.40, "loop_two_threshold": 0.20},
    {"dependency_threshold": 0.30, "and_threshold": 0.50, "loop_two_threshold": 0.30},
    {"dependency_threshold": 0.40, "and_threshold": 0.60, "loop_two_threshold": 0.40},
]

inductive_settings = [
    {"noise_threshold": 0.00},
    {"noise_threshold": 0.05},
    {"noise_threshold": 0.10},
]

DECISION_METRICS = ["fitness_log", "precision", "generalization", "simplicity"]


def evaluate_net(net, im, fm, eval_df):
    fit = pm4py.fitness_token_based_replay(
        eval_df, net, im, fm,
        activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
    )
    metrics = {
        "fitness_log": fit.get("log_fitness"),
        "fitness_average_trace": fit.get("average_trace_fitness"),
        "fitting_traces_percent": fit.get("percentage_of_fitting_traces"),
        "precision": pm4py.precision_token_based_replay(
            eval_df, net, im, fm,
            activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
        ),
        "generalization": pm4py.generalization_tbr(
            eval_df, net, im, fm,
            activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
        ),
        "simplicity": pm4py.simplicity_petri_net(net, im, fm),
        "f_score": None,
        "avg_score_percent": None,
    }
    f, p = metrics["fitness_log"], metrics["precision"]
    if f is not None and p is not None and (f + p) > 0:
        metrics["f_score"] = 2 * f * p / (f + p)
    available = [metrics[k] for k in DECISION_METRICS if metrics[k] is not None]
    if available:
        metrics["avg_score_percent"] = round(100 * sum(available) / len(available), 2)
    return metrics


def reduce_net(net, im, fm):
    net = pm4py.reduce_petri_net_invisibles(net)
    net, im, fm = pm4py.reduce_petri_net_implicit_places(net, im, fm)
    return net, im, fm


results = []

for sp_name, sp in SUBPROCESSES.items():
    sp_df = df[df[ACTIVITY].str.startswith(tuple(sp["prefixes"]))].copy()

    if sp.get("end"):
        last_act = (
            sp_df.sort_values([CASE_ID, TIMESTAMP])
            .groupby(CASE_ID)[ACTIVITY].last()
        )
        end_pattern = "|".join(re.escape(e) for e in sp["end"])
        keep_cases = last_act[last_act.str.contains(end_pattern, regex=True)].index
        sp_df = sp_df[sp_df[CASE_ID].isin(keep_cases)].copy()

    sp_df = sp_df.sort_values([CASE_ID, TIMESTAMP, ACTIVITY]).reset_index(drop=True)
    if sp_df.empty:
        continue

    all_cases = (
        pd.Series(sp_df[CASE_ID].unique()).sample(frac=1, random_state=42).tolist()
    )
    total_cases = len(all_cases)

    case_sample_sizes = [x for x in BASE_SAMPLE_SIZES if x < total_cases]
    case_sample_sizes.append(total_cases)

    for sample_size in case_sample_sizes:
        selected_cases = all_cases[:sample_size]
        eval_df = sp_df[sp_df[CASE_ID].isin(selected_cases)].copy()
        eval_df = eval_df.sort_values(
            [CASE_ID, TIMESTAMP, ACTIVITY]
        ).reset_index(drop=True)

        for setting in heuristics_settings:
            label = (
                sp_name + "_heuristics_cases_" + str(sample_size)
                + "_dep_" + str(setting["dependency_threshold"]).replace(".", "_")
                + "_and_" + str(setting["and_threshold"]).replace(".", "_")
                + "_loop2_" + str(setting["loop_two_threshold"]).replace(".", "_")
            )
            net, im, fm = pm4py.discover_petri_net_heuristics(eval_df, **setting)
            net, im, fm = reduce_net(net, im, fm)
            metrics = evaluate_net(net, im, fm, eval_df)
            pm4py.write_pnml(net, im, fm, os.path.join(OUTPUT_FOLDER, label + ".pnml"))
            results.append({
                "subprocess": sp_name,
                "algorithm": "Heuristics Miner",
                "sample_cases": sample_size,
                "sample_events": len(eval_df),
                "dependency_threshold": setting["dependency_threshold"],
                "and_threshold": setting["and_threshold"],
                "loop_two_threshold": setting["loop_two_threshold"],
                "noise_threshold": None,
                **metrics,
            })

        for setting in inductive_settings:
            label = (
                sp_name + "_inductive_cases_" + str(sample_size)
                + "_noise_" + str(setting["noise_threshold"]).replace(".", "_")
            )
            net, im, fm = pm4py.discover_petri_net_inductive(eval_df, **setting)
            net, im, fm = reduce_net(net, im, fm)
            metrics = evaluate_net(net, im, fm, eval_df)
            pm4py.write_pnml(net, im, fm, os.path.join(OUTPUT_FOLDER, label + ".pnml"))
            results.append({
                "subprocess": sp_name,
                "algorithm": "Inductive Miner",
                "sample_cases": sample_size,
                "sample_events": len(eval_df),
                "dependency_threshold": None,
                "and_threshold": None,
                "loop_two_threshold": None,
                "noise_threshold": setting["noise_threshold"],
                **metrics,
            })

results_df = pd.DataFrame(results)
results_path = os.path.join(OUTPUT_FOLDER, "permitlog_mining_quality_results.csv")
results_df.to_csv(results_path, index=False)

scored = results_df.dropna(subset=["avg_score_percent"])
best = scored.loc[scored.groupby("subprocess")["avg_score_percent"].idxmax()].sort_values(
    "avg_score_percent", ascending=False
)
best

replaying log with TBR, completed traces :: 100%|██████████| 156/156 [00:00<00:00, 1659.53it/s]


,subprocess,algorithm,sample_cases,sample_events,dependency_threshold,and_threshold,loop_two_threshold,noise_threshold,fitness_log,fitness_average_trace,fitting_traces_percent,precision,generalization,simplicity,f_score,avg_score_percent
19,permit,Heuristics Miner,7046,24102,0.3,0.5,0.3,NaN,0.951598,0.949349,54.016463,0.992588,0.769733,0.591398,0.971661,82.63
40,declaration,Inductive Miner,1000,4313,NaN,NaN,NaN,0.05,0.990203,0.988849,89.900000,0.606929,0.902215,0.666667,0.752578,79.15
53,request_for_payment,Inductive Miner,250,1283,NaN,NaN,NaN,0.10,0.991293,0.992601,87.600000,0.634104,0.831411,0.705882,0.773451,79.07


In [5]:
scored = results_df.dropna(subset=["avg_score_percent"])
best_per_algo = scored.loc[
    scored.groupby(["subprocess", "algorithm"])["avg_score_percent"].idxmax()
].sort_values(["subprocess", "algorithm"])

for _, row in best_per_algo.iterrows():
    scope = row["subprocess"]
    n = int(row["sample_cases"])
    if row["algorithm"] == "Heuristics Miner":
        label = (f"{scope}_heuristics_cases_{n}"
                 f"_dep_{str(float(row['dependency_threshold'])).replace('.', '_')}"
                 f"_and_{str(float(row['and_threshold'])).replace('.', '_')}"
                 f"_loop2_{str(float(row['loop_two_threshold'])).replace('.', '_')}")
    else:
        label = (f"{scope}_inductive_cases_{n}"
                 f"_noise_{str(float(row['noise_threshold'])).replace('.', '_')}")
    pnml_path = os.path.join(OUTPUT_FOLDER, label + ".pnml")
    png_path = os.path.join(OUTPUT_FOLDER, label + ".png")
    net, im, fm = pm4py.read_pnml(pnml_path)
    pm4py.save_vis_petri_net(net, im, fm, png_path)

best_per_algo

,subprocess,algorithm,sample_cases,sample_events,dependency_threshold,and_threshold,loop_two_threshold,noise_threshold,fitness_log,fitness_average_trace,fitting_traces_percent,precision,generalization,simplicity,f_score,avg_score_percent
42,declaration,Heuristics Miner,5571,24218,0.2,0.4,0.2,NaN,0.957997,0.955965,54.837552,0.943899,0.699060,0.539568,0.950896,78.51
40,declaration,Inductive Miner,1000,4313,NaN,NaN,NaN,0.05,0.990203,0.988849,89.900000,0.606929,0.902215,0.666667,0.752578,79.15
19,permit,Heuristics Miner,7046,24102,0.3,0.5,0.3,NaN,0.951598,0.949349,54.016463,0.992588,0.769733,0.591398,0.971661,82.63
4,permit,Inductive Miner,250,857,NaN,NaN,NaN,0.05,0.995898,0.996524,97.200000,0.759079,0.796557,0.702128,0.861510,81.34
60,request_for_payment,Heuristics Miner,1000,5004,0.2,0.4,0.2,NaN,0.842610,0.828321,1.400000,0.964337,0.741703,0.553191,0.899373,77.55
53,request_for_payment,Inductive Miner,250,1283,NaN,NaN,NaN,0.10,0.991293,0.992601,87.600000,0.634104,0.831411,0.705882,0.773451,79.07
